# Higgs Boson Detection with TPUs - Exploration Notebook

This notebook provides an interactive exploration of the Higgs Boson detection pipeline using Google TPUs.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns

# Check TPU availability
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    print(f"Connected to TPU: {tpu.master()}")
    strategy = tf.distribute.TPUStrategy(tpu)
except ValueError:
    print("TPU not available, using CPU/GPU")
    strategy = tf.distribute.get_strategy()

print(f"Number of replicas: {strategy.num_replicas_in_sync}")

In [ ]:
# Import project modules
import sys
sys.path.insert(0, '..')

from src.data_loader import HiggsDataset, create_sample_data
from src.model import HiggsClassifier, ResNetHiggs, DenseNetHiggs, create_model
from src.evaluate import AMSMetric, calculate_ams, evaluate_model
from src.train import Trainer

## Generate Sample Data for Testing

In [ ]:
# Create synthetic data for demonstration
X, y = create_sample_data(n_samples=10000, n_features=28)
print(f"Generated {len(X)} samples with {X.shape[1]} features")
print(f"Signal fraction: {np.mean(y):.2%}")

In [ ]:
# Visualize feature distributions
fig, axes = plt.subplots(4, 7, figsize=(20, 12))
axes = axes.flatten()

for i in range(28):
    signal_mask = y == 1
    background_mask = y == 0
    
    axes[i].hist(X[background_mask, i], bins=50, alpha=0.5, label='Background', density=True)
    axes[i].hist(X[signal_mask, i], bins=50, alpha=0.5, label='Signal', density=True)
    axes[i].set_xlabel(f'Feature {i}')
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.show()

## Create and Test Models

In [ ]:
# Split data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Create TensorFlow datasets
BATCH_SIZE = 256

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Test MLP Model
with strategy.scope():
    mlp = create_model('mlp', input_dim=28, hidden_dims=[256, 128, 64])
    mlp.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', AMSMetric()]
    )

print(f"MLP Parameters: {mlp.count_params():,}")
mlp.summary()

In [ ]:
# Train MLP
history_mlp = mlp.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    verbose=1
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history_mlp.history['loss'], label='Train Loss')
axes[0].plot(history_mlp.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].set_title('Binary Crossentropy Loss')

# Accuracy
axes[1].plot(history_mlp.history['accuracy'], label='Train Acc')
axes[1].plot(history_mlp.history['val_accuracy'], label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].set_title('Classification Accuracy')

# AMS
if 'ams' in history_mlp.history:
    axes[2].plot(history_mlp.history['ams'], label='Train AMS')
    axes[2].plot(history_mlp.history['val_ams'], label='Val AMS')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('AMS Score')
    axes[2].legend()
    axes[2].set_title('Approximate Median Significance')

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
print("\n=== MLP Evaluation ===")
results = evaluate_model(mlp, test_ds)

# Get predictions for visualization
y_pred_prob = mlp.predict(test_ds, verbose=0).flatten()
y_pred_binary = (y_pred_prob >= 0.5).astype(int)

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix, roc_curve, auc

cm = confusion_matrix(y_test, y_pred_binary)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Background', 'Signal'],
            yticklabels=['Background', 'Signal'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

## Compare Different Architectures

In [ ]:
# Quick comparison of architectures
models_to_compare = ['mlp', 'resnet', 'densenet']
results_comparison = []

for model_type in models_to_compare:
    print(f"\nTraining {model_type.upper()}...")
    
    with strategy.scope():
        if model_type == 'mlp':
            model = create_model(model_type, input_dim=28, hidden_dims=[128, 64])
        elif model_type == 'resnet':
            model = create_model(model_type, input_dim=28, num_blocks=2, units_per_block=64)
        else:  # densenet
            model = create_model(model_type, input_dim=28, growth_rate=16, num_dense_blocks=2)
        
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    # Train briefly for comparison
    history = model.fit(train_ds, validation_data=val_ds, epochs=5, verbose=0)
    
    # Evaluate
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    
    results_comparison.append({
        'Model': model_type.upper(),
        'Parameters': f"{model.count_params():,}",
        'Test Accuracy': f"{test_acc:.4f}",
        'Final Val Loss': f"{history.history['val_loss'][-1]:.4f}"
    })

# Display comparison table
comparison_df = pd.DataFrame(results_comparison)
display(comparison_df)

## Next Steps

1. **Download Full Dataset**: Get the complete HIGGS dataset from UCI ML Repository
2. **Hyperparameter Tuning**: Use Keras Tuner or Optuna for optimization
3. **Full TPU Training**: Scale up to full dataset with TPU acceleration
4. **Ensemble Methods**: Combine multiple models for better performance
5. **Feature Engineering**: Add domain-specific features from particle physics